In [1]:
import pandas as pd
import numpy as np
import re
import os
from datetime import datetime
import src.datetime_utils as dateTime

from src.EqCat import EqCat

In [2]:
# функция для загрузки каталога

def load_isc_catalog(file_path):

    # Read all lines from file
    with open(file_path, 'r') as f:
        lines = f.readlines()
    
    # Find data section (skip header and stop at 'STOP')
    start_idx = 21  # Skip first 21 header lines
    end_idx = len(lines)
    
    # Find where data ends (at 'STOP')
    for i in range(start_idx, len(lines)):
        if lines[i].strip().upper() == 'STOP':
            end_idx = i
            break
    
    # Process data lines
    events = []
    magnitudes = []
    
    for i in range(start_idx, end_idx):
        line = lines[i].strip()
        if not line or line.startswith('#'):
            continue
            
        # Remove trailing comma if present
        if line.endswith(','):
            line = line[:-1]
            
        # Split by comma
        parts = [part.strip() for part in line.split(',')]
        
        # Skip malformed lines
        if len(parts) < 9 or not parts[0].replace('.', '').isdigit():
            continue
        
        try:
            # Extract basic event information (first 9 fields)
            event_id = int(parts[0])
            event_type = parts[1].strip()
            author = parts[2].strip()
            date = parts[3].strip()
            time = parts[4].strip()
            
            # Parse coordinates and depth
            lat = float(parts[5]) if parts[5] else np.nan
            lon = float(parts[6]) if parts[6] else np.nan
            depth = float(parts[7]) if parts[7] else np.nan
            depfix = parts[8].strip()
            
            # Add to events list (ONLY basic event data, no magnitudes)
            events.append({
                'EVENTID': event_id,
                'TYPE': event_type,
                'AUTHOR': author,
                'DATE': date,
                'TIME': time,
                'LAT': lat,
                'LON': lon,
                'DEPTH': depth,
                'DEPFIX': depfix
            })
            
            # Process ALL magnitudes starting from position 9 (groups of 3)
            j = 9
            while j + 2 < len(parts):
                mag_author = parts[j].strip()
                mag_type = parts[j+1].strip()
                mag_value = parts[j+2].strip()
                
                # Only add valid magnitude values
                if mag_value and re.match(r'-?\d+(\.\d+)?', mag_value):
                    magnitudes.append({
                        'EVENTID': event_id,
                        'AUTHOR': mag_author,
                        'MAG_TYPE': mag_type,
                        'MAG_VALUE': float(mag_value)
                    })
                
                j += 3
                
        except (ValueError, IndexError) as e:
            print(f"Warning: Error parsing line {i+1}: {e}")
            print(f"Line content: {line[:100]}...")
    
    # Create DataFrames
    main_df = pd.DataFrame(events)
    magnitudes_df = pd.DataFrame(magnitudes) if magnitudes else pd.DataFrame(columns=['EVENTID', 'AUTHOR', 'MAG_TYPE', 'MAG_VALUE'])
    
    print(f"Successfully loaded {len(main_df)} earthquake events")
    if not magnitudes_df.empty:
        print(f"Loaded {len(magnitudes_df)} magnitude measurements from {magnitudes_df['AUTHOR'].nunique()} agencies")
    
    return main_df, magnitudes_df

In [3]:
file_path = 'data/Chignik_eqs.txt'

main_df, magnitudes_df = load_isc_catalog(file_path)

neic_magnitudes = magnitudes_df[magnitudes_df['AUTHOR'] == 'NEIC'].copy()

neic_magnitudes = neic_magnitudes.groupby('EVENTID').first().reset_index()

merged_df = pd.merge(main_df, neic_magnitudes, on='EVENTID', how='inner')

merged_df['time'] = pd.to_datetime(
    merged_df['DATE'] + ' ' + merged_df['TIME'].str.replace(',', ' '),
    errors='coerce'
)

print(merged_df)

def datetime_to_decimal_year(dt_series):
    def to_dec_year(t):
        if pd.isna(t):
            return np.nan
        return dateTime.dateTime2decYr([t.year, t.month, t.day, t.hour, t.minute, t.second])
    return np.array([to_dec_year(t) for t in dt_series])

eqCat = EqCat( )
n_events = len(merged_df)
eqCat.data = {
    'Time': datetime_to_decimal_year(merged_df['time']),
    'N': merged_df['EVENTID'].values,
    'Lat': merged_df['LAT'].values,
    'Lon': merged_df['LON'].values,
    'Depth': merged_df['DEPTH'].values,
    'Mag': merged_df['MAG_VALUE'].values
}

output_file = 'data/Chignik_eqs.mat'
eqCat.saveMatBin(output_file)

Successfully loaded 842 earthquake events
Loaded 2876 magnitude measurements from 13 agencies
       EVENTID TYPE AUTHOR_x        DATE         TIME      LAT       LON  \
0    622450815   se     NEIC  2021-07-29  05:49:38.53  55.4417 -162.4712   
1    620857937   ke      ISC  2021-07-29  06:15:47.49  55.4449 -157.9970   
2    622450816   se      ISC  2021-07-29  06:19:21.75  55.0062 -158.1592   
3    622450817   se     NEIC  2021-07-29  06:20:44.45  55.2218 -157.7952   
4    626337071   se      ISC  2021-07-29  06:23:01.82  55.8975 -157.1073   
..         ...  ...      ...         ...          ...      ...       ...   
834  622499435   se     NEIC  2021-09-27  08:55:59.86  55.4032 -158.2236   
835  622499437   se     NEIC  2021-09-27  13:01:38.54  54.6052 -159.7268   
836  622499444   se     NEIC  2021-09-27  22:25:39.78  56.8884 -155.4297   
837  622499450   se     NEIC  2021-09-29  01:12:15.96  54.4273 -159.7855   
838  622499456   se     NEIC  2021-09-29  05:13:34.36  54.7830 -156.94